In [ ]:
# ---------------------------------------------------------------------------
# Environment setup — works on both Colab and local
# ---------------------------------------------------------------------------
import sys, os, pathlib

# Fix sympy/torchvision compatibility on Colab (Python 3.12)
import sympy
import sympy.printing  # must be imported before torchvision loads

# Find the pnpflow package directory regardless of where the kernel started
# Strategy: walk up from the notebook file (if available) or cwd until we find pnpflow/
def _find_pnpflow_root():
    """Return the repo root that contains the 'pnpflow' package directory."""
    # On Colab after git clone, the repo lives under /content/
    candidates = []

    # 1. Try relative to this notebook's known location in the repo (pnpflow/methods/)
    #    Works locally when cwd is the repo root or the notebook dir.
    for start in [pathlib.Path.cwd(), pathlib.Path.cwd().parent, pathlib.Path.cwd().parent.parent]:
        if (start / 'pnpflow' / 'blind_degradations.py').exists():
            return str(start)

    # 2. Colab: after `!git clone <repo>` into /content/
    for d in pathlib.Path('/content').iterdir() if pathlib.Path('/content').exists() else []:
        if d.is_dir() and (d / 'pnpflow' / 'blind_degradations.py').exists():
            return str(d)

    raise FileNotFoundError(
        "Could not locate the pnpflow package. "
        "On Colab, clone the repo first:\n"
        "  !git clone https://github.com/<user>/COMP0138-FYP.git\n"
        "Locally, launch Jupyter from the repo root."
    )

ROOT = _find_pnpflow_root()
PNPFLOW = os.path.join(ROOT, 'pnpflow')

for p in [ROOT, PNPFLOW]:
    if p not in sys.path:
        sys.path.insert(0, p)

print(f"Repo root : {ROOT}")
print(f"pnpflow   : {PNPFLOW}")

# --- imports from pnpflow package ---
from blind_degradations import LearnableGaussianBlur
from blind_data import make_blind_gaussian_blur_problem, BlindGaussianBlurProblem, operator_mse
from utils import define_model, load_model
from degradations import GaussianDeblurring
from dataloaders import DataLoaders

import torch, numpy as np, matplotlib.pyplot as plt, math, types
from typing import Dict, List

print('All imports successful!')

In [ ]:
# ---------------------------------------------------------------------------
# CONFIG — edit these to match your environment
# ---------------------------------------------------------------------------

DEVICE       = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_PATH   = os.path.join(ROOT, 'model', 'celeba', 'ot', 'model_final.pt')
DATA_ROOT    = os.path.join(ROOT, 'data', 'celeba')
IMG_SIZE     = 128
NUM_CHANNELS = 3

torch.manual_seed(0)
print(f'Device: {DEVICE}')
print(f'Model path: {MODEL_PATH}  (exists: {os.path.isfile(MODEL_PATH)})')
print(f'Data root:  {DATA_ROOT}  (exists: {os.path.isdir(DATA_ROOT)})')

In [ ]:
# Load OT Flow Matching model
model_args = types.SimpleNamespace(model='ot', num_channels=NUM_CHANNELS, dim_image=IMG_SIZE)
model, _ = define_model(model_args)
load_model('ot', model, None, download=False, checkpoint_path=MODEL_PATH, dataset=None, device=DEVICE)
model = model.to(DEVICE)
model.eval()

n_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {n_params/1e6:.1f}M parameters')

In [ ]:
# Load N=32 CelebA test images (enough for B=32 multi-image experiments)
# dataloaders.py stores relative paths at construction time but resolves
# them at __getitem__ time, so we must make them absolute.
import importlib, dataloaders as _dl_mod
importlib.reload(_dl_mod)
from dataloaders import DataLoaders, CelebADataset

# Patch CelebADataset.__init__ to absolutize img_dir
_orig_init = CelebADataset.__init__

def _abs_init(self, img_dir, partition_csv, partition, transform=None):
    _orig_init(self, os.path.abspath(img_dir), os.path.abspath(partition_csv), partition, transform)

CelebADataset.__init__ = _abs_init

_prev_cwd = os.getcwd()
os.chdir(ROOT)

N_IMAGES = 8
dl = DataLoaders('celeba', batch_size_train=1, batch_size_test=1)
test_loader = dl.load_data()['test']

os.chdir(_prev_cwd)
CelebADataset.__init__ = _orig_init  # restore

x_gts = []
for i, (x, _) in enumerate(test_loader):
    if i >= N_IMAGES:
        break
    x_gts.append(x.to(DEVICE))

print(f'Loaded {len(x_gts)} images, shape {x_gts[0].shape}')

# Show first 8 images
fig, axes = plt.subplots(1, 8, figsize=(16, 2.5))
for ax, x in zip(axes, x_gts[:8]):
    img = (x[0].cpu().clamp(-1, 1) + 1) / 2
    ax.imshow(img.permute(1, 2, 0).numpy())
    ax.axis('off')
plt.suptitle(f'CelebA test images (showing 8 of {len(x_gts)})')
plt.tight_layout()
plt.show()

In [ ]:
def postprocess(x: torch.Tensor) -> torch.Tensor:
    """[-1,1] -> [0,1] for display/PSNR."""
    return (x.clamp(-1, 1) + 1) / 2


def psnr_db(x_hat: torch.Tensor, x_gt: torch.Tensor) -> float:
    """PSNR in dB between two tensors in [-1,1]."""
    mse = torch.mean((postprocess(x_hat) - postprocess(x_gt)) ** 2).item()
    if mse < 1e-12:
        return float('inf')
    return 10 * math.log10(1.0 / mse)


def viz_row(tensors: list, titles: list = None, save: str = None):
    """Show a row of images from [-1,1] tensors."""
    n = len(tensors)
    fig, axes = plt.subplots(1, n, figsize=(3 * n, 3.2))
    if n == 1:
        axes = [axes]
    for ax, t, title in zip(axes, tensors, titles or [''] * n):
        img = postprocess(t[0]).cpu().permute(1, 2, 0).numpy()
        ax.imshow(img.squeeze(), cmap='gray' if img.shape[-1] == 1 else None)
        ax.set_title(title, fontsize=9)
        ax.axis('off')
    plt.tight_layout()
    if save:
        plt.savefig(save, bbox_inches='tight')
    plt.show()


def plot_loss_sigma(sigma_hist: list, loss_hist: list, sigma_true: float,
                    title: str = '', save: str = None):
    """Two-panel: sigma convergence | data loss."""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 3.5))
    iters = range(len(sigma_hist))
    ax1.plot(iters, sigma_hist, label='sigma_hat')
    ax1.axhline(sigma_true, color='r', linestyle='--', label=f'sigma_true={sigma_true}')
    ax1.set_xlabel('Iteration'); ax1.set_ylabel('sigma'); ax1.legend(); ax1.set_title('Sigma convergence')
    ax2.semilogy(iters, loss_hist)
    ax2.set_xlabel('Iteration'); ax2.set_ylabel('Data loss'); ax2.set_title('Data-consistency loss')
    plt.suptitle(title)
    plt.tight_layout()
    if save:
        plt.savefig(save, bbox_inches='tight')
    plt.show()


def plot_sigma_bars(Bs: list, sigma_finals: list, sigma_true: float, title: str = ''):
    """Bar chart of sigma_final vs number of images B."""
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.bar([str(b) for b in Bs], sigma_finals, color='steelblue', alpha=0.8)
    ax.axhline(sigma_true, color='r', linestyle='--', label=f'sigma_true={sigma_true}')
    ax.set_xlabel('Batch size B'); ax.set_ylabel('sigma_final')
    ax.set_title(title); ax.legend()
    plt.tight_layout()
    plt.show()


print('Helpers defined.')

In [ ]:
# --- PnP-Flow building blocks ---

def _model_velocity(model, x, t, model_type='ot'):
    """Return v_theta(x, t)."""
    if model_type == 'ot':
        return model(x, t)
    elif model_type == 'rectified':
        import pnpflow.image_generation.models.utils as mutils
        model_fn = mutils.get_model_fn(model, train=False)
        return model_fn(x.float(), t * 999)
    else:
        raise ValueError(f"Unknown model_type '{model_type}'.")


def _lr_schedule(lr, t, style, alpha=1.0):
    """Step-size schedule gamma(t)."""
    if style == '1_minus_t':         return lr * (1.0 - t)
    elif style == 'sqrt_1_minus_t':  return lr * (1.0 - t) ** 0.5
    elif style == 'alpha_1_minus_t': return lr * (1.0 - t) ** alpha
    elif style == 'constant':        return lr
    else: raise ValueError(f"Unknown gamma_style '{style}'.")


def _interpolate(x, t):
    """x_tilde = t*x + (1-t)*eps, eps ~ N(0,I)."""
    t_vec = t.view(-1, 1, 1, 1)
    return t_vec * x + (1.0 - t_vec) * torch.randn_like(x)


def _denoiser(model, x, t, model_type='ot'):
    """D(x,t) = x + (1-t)*v_theta(x,t)."""
    t_vec = t.view(-1, 1, 1, 1)
    v = _model_velocity(model, x, t, model_type)
    return x + (1.0 - t_vec) * v


print('Building blocks defined.')

In [ ]:
def _pnp_flow_trajectory_blind(
    model, *, x_init, y, op, opt_sigma, sigma_noise, num_steps, lr,
    gamma_style='1_minus_t', alpha=1.0, num_samples=1, model_type='ot',
    update_kernel=True, clip_grad_norm=1.0,
    sigma_update_until_t=1.0, sigma_history=None,
):
    """
    One full PnP-Flow trajectory (t: 0->1) with per-step kernel update.

    As described in the paper (A.14):
      For each time step t:
        1. Data-fit gradient step:  z = x - lr(t) * H(H(x) - y)
        2. Denoiser step:           x = D(interp(z, t), t)
        3. Kernel update:           optimizer step on ||y - H_learned(x)||^2
           (only if t < sigma_update_until_t)

    Args:
        sigma_update_until_t: Only update sigma for t < this value.
            The loss landscape is well-posed at early/intermediate t;
            at later t the denoiser removes blur info, shifting the
            minimum below sigma_true.  Default 1.0 = update everywhere.
        sigma_history: Optional list to append sigma values after each step.
    """
    device = x_init.device
    x = x_init.clone()
    delta = 1.0 / num_steps

    for k in range(num_steps):
        t_scalar = delta * k
        t = torch.full((len(x),), t_scalar, device=device)

        # Step 1: data-fit gradient
        with torch.no_grad():
            residual = op(x) - y
            grad     = op(residual)
            lr_k     = _lr_schedule(lr, t_scalar, gamma_style, alpha)
            z        = x - lr_k * grad

        # Step 2: denoiser
        with torch.no_grad():
            x_new = torch.zeros_like(x)
            for _ in range(num_samples):
                z_tilde = _interpolate(z, t)
                x_new   = x_new + _denoiser(model, z_tilde, t, model_type)
            x = x_new / num_samples

        # Step 3: kernel update (only at early time steps)
        if update_kernel and opt_sigma is not None and t_scalar < sigma_update_until_t:
            opt_sigma.zero_grad(set_to_none=True)
            loss_kernel = torch.mean((y - op(x.detach())) ** 2)
            loss_kernel.backward()
            if clip_grad_norm is not None:
                torch.nn.utils.clip_grad_norm_(op.parameters(), clip_grad_norm)
            opt_sigma.step()
            op.clamp_params_()

        if sigma_history is not None:
            sigma_history.append(op.sigma().item())

    return x


# Keep the non-blind version too (for Aim 3)
def _pnp_flow_trajectory(
    model, *, x_init, y, op, sigma_noise, num_steps, lr,
    gamma_style='1_minus_t', alpha=1.0, num_samples=1, model_type='ot',
):
    """Non-blind trajectory (no kernel update)."""
    return _pnp_flow_trajectory_blind(
        model, x_init=x_init, y=y, op=op, opt_sigma=None,
        sigma_noise=sigma_noise, num_steps=num_steps, lr=lr,
        gamma_style=gamma_style, alpha=alpha, num_samples=num_samples,
        model_type=model_type, update_kernel=False,
    )


print('_pnp_flow_trajectory and _pnp_flow_trajectory_blind defined.')

In [ ]:
def _search_sigma(x, y, sigma_max, n_candidates=30, refine=True):
    """
    Find sigma that minimises ||H_sigma(x) - y||^2 via grid search
    with parabolic refinement around the minimum.
    """
    device = x.device
    sigmas = torch.linspace(0.1, sigma_max, n_candidates, device=device)
    losses = []
    for s in sigmas:
        op_tmp = LearnableGaussianBlur(init_sigma=s.item(), sigma_max=sigma_max,
                                       padding='reflect')
        op_tmp = op_tmp.to(device=device, dtype=x.dtype)
        with torch.no_grad():
            loss = torch.mean((op_tmp(x) - y) ** 2).item()
        losses.append(loss)

    best_idx = min(range(len(losses)), key=lambda i: losses[i])

    if refine and 0 < best_idx < len(sigmas) - 1:
        s1, s2, s3 = (sigmas[best_idx-1].item(), sigmas[best_idx].item(),
                       sigmas[best_idx+1].item())
        l1, l2, l3 = losses[best_idx-1], losses[best_idx], losses[best_idx+1]
        denom = 2 * ((s2-s1)*(l2-l3) - (s2-s3)*(l2-l1))
        if abs(denom) > 1e-12:
            s_opt = s2 - ((s2-s1)**2*(l2-l3) - (s2-s3)**2*(l2-l1)) / denom
            s_opt = max(0.1, min(s_opt, sigma_max))
            return s_opt
    return sigmas[best_idx].item()


def _pnp_flow_trajectory_search_once(
    model, *, x_init, y, op, sigma_max, num_steps, lr,
    gamma_style='1_minus_t', alpha=1.0, num_samples=1, model_type='ot',
    search_at_t=0.2, n_candidates=30, verbose=False,
):
    """
    PnP-Flow trajectory that estimates sigma ONCE via grid search at a
    specific flow time t, then freezes sigma for the rest.

    Key insight: at intermediate t (~0.2), x is partially deblurred and
    the loss landscape ||H_sigma(x)-y||^2 has a well-defined minimum near
    sigma_true. At later t the minimum shifts toward 0 (identity trap).
    """
    device = x_init.device
    x = x_init.clone()
    delta = 1.0 / num_steps
    searched = False

    for k in range(num_steps):
        t_scalar = delta * k
        t = torch.full((len(x),), t_scalar, device=device)

        with torch.no_grad():
            residual = op(x) - y
            grad     = op(residual)
            lr_k     = _lr_schedule(lr, t_scalar, gamma_style, alpha)
            z        = x - lr_k * grad

        with torch.no_grad():
            x_new = torch.zeros_like(x)
            for _ in range(num_samples):
                z_tilde = _interpolate(z, t)
                x_new   = x_new + _denoiser(model, z_tilde, t, model_type)
            x = x_new / num_samples

        if not searched and t_scalar >= search_at_t:
            with torch.no_grad():
                sigma_opt = _search_sigma(x, y, sigma_max,
                                          n_candidates=n_candidates)
                op.set_sigma_(sigma_opt)
                searched = True
                if verbose:
                    print(f'  step {k:3d}/{num_steps}  t={t_scalar:.3f}  '
                          f'sigma={sigma_opt:.4f}')

    return x


print('_search_sigma, _pnp_flow_trajectory_search_once defined.')

In [ ]:
# Setup blind problems for ALL test images
SIGMA_TRUE  = 1.5
SIGMA_NOISE = 0.05
SIGMA_INIT  = 3.0
SIGMA_MAX   = 5.0

probs = []
for i, x_gt in enumerate(x_gts):
    torch.manual_seed(i)
    prob = make_blind_gaussian_blur_problem(
        x_gt=x_gt.to(DEVICE),
        sigma_true=SIGMA_TRUE,
        sigma_noise=SIGMA_NOISE,
        sigma_max=SIGMA_MAX,
        padding='reflect',
        seed=i,
    )
    probs.append(prob)

# Keep first one for backward compat
prob0 = probs[0]
x_gt0 = x_gts[0].to(DEVICE)

print(f'Created {len(probs)} blind problems')
print(f'sigma_true = {SIGMA_TRUE}, sigma_init = {SIGMA_INIT}')
viz_row([x_gt0, prob0.y], titles=['Clean x_gt', 'Observation y'])

In [ ]:
def _estimate_sigma_search(
    model, *, y, sigma_init, sigma_max, pnp_steps, pnp_lr,
    gamma_style, alpha, num_samples, model_type,
    search_at_t=0.2, n_candidates=30,
):
    """
    Run a partial PnP-Flow trajectory from y and search for sigma at t.
    Returns the sigma estimate (a single float).
    """
    device = y.device
    op = LearnableGaussianBlur(init_sigma=sigma_init, sigma_max=sigma_max,
                               padding='reflect').to(device=device, dtype=y.dtype)
    x = y.clone().detach()
    delta = 1.0 / pnp_steps

    for k in range(pnp_steps):
        t_scalar = delta * k
        t = torch.full((1,), t_scalar, device=device)

        with torch.no_grad():
            residual = op(x) - y
            grad = op(residual)
            lr_k = _lr_schedule(pnp_lr, t_scalar, gamma_style, alpha)
            z = x - lr_k * grad

            x_new = torch.zeros_like(x)
            for _ in range(num_samples):
                z_tilde = _interpolate(z, t)
                x_new = x_new + _denoiser(model, z_tilde, t, model_type)
            x = x_new / num_samples

        if t_scalar >= search_at_t:
            with torch.no_grad():
                return _search_sigma(x, y, sigma_max, n_candidates=n_candidates)

    return sigma_init  # fallback


def blind_pnp_flow_robust(
    prob: BlindGaussianBlurProblem,
    *, model, device,
    sigma_init: float, sigma_max: float,
    n_estimation_passes: int = 5,
    adam_iters: int = 10,
    pnp_steps: int = 50, pnp_lr: float = 1.0,
    sigma_noise: float = 0.05,
    gamma_style: str = '1_minus_t', alpha: float = 1.0,
    num_samples: int = 1, model_type: str = 'ot',
    search_at_t: float = 0.2, n_candidates: int = 30,
    operator_lr: float = 1e-3, clip_grad_norm: float = 1.0,
    sigma_update_until_t: float = 1.0,
    print_every: int = 1,
) -> Dict:
    """
    Robust search-initialised blind PnP-Flow (NOVEL).

    1. Multi-pass estimation: run N partial trajectories from y,
       each with independent noise, search for sigma at t=search_at_t.
       Take the MEDIAN as a robust estimate.
    2. Adaptive margin: start Adam from median + 0.5 (ensures we are
       above sigma_true without requiring prior knowledge of sigma range).
    3. Adam refinement: per-step Adam from the informed starting point.
    """
    y = prob.y.to(device)

    common_est = dict(
        model=model, y=y, sigma_init=sigma_init, sigma_max=sigma_max,
        pnp_steps=pnp_steps, pnp_lr=pnp_lr, gamma_style=gamma_style,
        alpha=alpha, num_samples=num_samples, model_type=model_type,
        search_at_t=search_at_t, n_candidates=n_candidates,
    )

    # Step 1: Multi-pass estimation with median
    estimates = []
    for p in range(n_estimation_passes):
        sigma_est = _estimate_sigma_search(**common_est)
        estimates.append(sigma_est)

    estimates_sorted = sorted(estimates)
    sigma_median = estimates_sorted[len(estimates_sorted) // 2]

    if print_every > 0:
        est_str = ', '.join(f'{e:.3f}' for e in estimates)
        print(f'[robust] estimates: [{est_str}]')
        print(f'[robust] median={sigma_median:.4f}')

    # Step 2: Adaptive margin - always start above the estimate
    sigma_start = min(sigma_median + 0.5, sigma_max)

    if print_every > 0:
        print(f'[robust] start={sigma_start:.4f} (median + 0.5)')

    # Step 3: Adam refinement
    out = blind_pnp_flow_single(
        prob, model=model, device=device,
        sigma_init=sigma_start, sigma_max=sigma_max,
        outer_iters=adam_iters, pnp_steps=pnp_steps, pnp_lr=pnp_lr,
        sigma_noise=sigma_noise, gamma_style=gamma_style, alpha=alpha,
        num_samples=num_samples, model_type=model_type,
        operator_lr=operator_lr, clip_grad_norm=clip_grad_norm,
        sigma_update_until_t=sigma_update_until_t,
        print_every=print_every,
    )

    out['sigma_estimates'] = estimates
    out['sigma_median'] = sigma_median
    out['sigma_start'] = sigma_start
    return out


def blind_pnp_flow_two_pass(
    prob: BlindGaussianBlurProblem,
    *, model, device,
    sigma_init: float, sigma_max: float,
    pnp_steps: int = 50, pnp_lr: float = 1.0,
    sigma_noise: float = 0.05,
    gamma_style: str = '1_minus_t', alpha: float = 1.0,
    num_samples: int = 1, model_type: str = 'ot',
    search_at_t: float = 0.2, n_candidates: int = 30,
    verbose: bool = True,
) -> Dict:
    """Two-pass blind PnP-Flow: search at t then reconstruct."""
    y = prob.y.to(device)

    sigma_est = _estimate_sigma_search(
        model=model, y=y, sigma_init=sigma_init, sigma_max=sigma_max,
        pnp_steps=pnp_steps, pnp_lr=pnp_lr, gamma_style=gamma_style,
        alpha=alpha, num_samples=num_samples, model_type=model_type,
        search_at_t=search_at_t, n_candidates=n_candidates,
    )
    if verbose:
        print(f'[two_pass] sigma_est={sigma_est:.4f}')

    op_final = LearnableGaussianBlur(init_sigma=sigma_est, sigma_max=sigma_max,
                                     padding='reflect')
    op_final = op_final.to(device=device, dtype=y.dtype)

    x = _pnp_flow_trajectory(
        model, x_init=y.clone(), y=y, op=op_final, sigma_noise=sigma_noise,
        num_steps=pnp_steps, lr=pnp_lr, gamma_style=gamma_style,
        alpha=alpha, num_samples=num_samples, model_type=model_type,
    ).detach()

    with torch.no_grad():
        loss_val = torch.mean((op_final(x) - y) ** 2).item()
    if verbose:
        print(f'[two_pass] sigma={sigma_est:.4f}  loss={loss_val:.4e}')

    return {'x': x, 'sigma_final': sigma_est,
            'sigma_history': [sigma_est], 'loss_history': [loss_val]}


def blind_pnp_flow_single(
    prob: BlindGaussianBlurProblem,
    *, model, device,
    sigma_init: float, sigma_max: float,
    outer_iters: int = 30,
    pnp_steps: int = 50, pnp_lr: float = 1.0,
    sigma_noise: float = 0.05,
    gamma_style: str = '1_minus_t', alpha: float = 1.0,
    num_samples: int = 1, model_type: str = 'ot',
    operator_lr: float = 1e-3, clip_grad_norm: float = 1.0,
    sigma_update_until_t: float = 1.0,
    print_every: int = 1,
) -> Dict:
    """
    Per-step Adam blind PnP-Flow (paper A.14 baseline).

    Args:
        sigma_update_until_t: Only update sigma for time steps t < this value.
            At later t, the denoiser removes blur info, biasing sigma downward.
            Use e.g. 0.3 to restrict updates to the well-posed region.
    """
    y  = prob.y.to(device)
    op = LearnableGaussianBlur(init_sigma=sigma_init, sigma_max=sigma_max,
                               padding='reflect')
    op = op.to(device=device, dtype=y.dtype)
    opt_sigma = torch.optim.Adam(op.parameters(), lr=operator_lr)
    x = y.clone().detach()

    sigma_hist, loss_hist = [], []
    sigma_step_hist = []  # per-step sigma within each trajectory

    for outer in range(outer_iters):
        x = _pnp_flow_trajectory_blind(
            model, x_init=x, y=y, op=op, opt_sigma=opt_sigma,
            sigma_noise=sigma_noise, num_steps=pnp_steps, lr=pnp_lr,
            gamma_style=gamma_style, alpha=alpha, num_samples=num_samples,
            model_type=model_type, update_kernel=True,
            clip_grad_norm=clip_grad_norm,
            sigma_update_until_t=sigma_update_until_t,
            sigma_history=sigma_step_hist,
        ).detach()

        with torch.no_grad():
            loss_val = torch.mean((op(x) - y) ** 2).item()
        sigma_hist.append(op.sigma().item())
        loss_hist.append(loss_val)

        if print_every > 0 and (outer % print_every == 0
                                or outer == outer_iters - 1):
            print(f'[adam {outer:03d}/{outer_iters}]  '
                  f'sigma={op.sigma().item():.4f}  loss={loss_val:.4e}')

    return {'x': x, 'sigma_final': op.sigma().item(),
            'sigma_history': sigma_hist, 'loss_history': loss_hist,
            'sigma_step_history': sigma_step_hist}


print('blind_pnp_flow_robust, blind_pnp_flow_two_pass, '
      'blind_pnp_flow_single defined.')